Retaining only "TRANSFER" and "CASH_OUT" columns

In [5]:
import pandas as pd 
data = pd.read_csv("../data/transaction_data.csv")
data = data[data["type"].isin(["TRANSFER", "CASH_OUT"])]

In [10]:
data.shape

(2770409, 11)

In [16]:
fraud_rate_per_transaction = data.groupby("type")["isFraud"].mean()
fraud_rate = data["isFraud"].mean()

print(f'fraud rate across dataset: {fraud_rate:.3%}')
print('fraud rates in TRANSFER and CASH_OUT transactions:')
print(fraud_rate_per_transaction)


fraud rate across dataset: 0.296%
fraud rates in TRANSFER and CASH_OUT transactions:
type
CASH_OUT    0.001840
TRANSFER    0.007688
Name: isFraud, dtype: float64


### train test split

In [34]:
from sklearn.model_selection import train_test_split

X = data[["step", "type", "amount", "oldbalanceOrg", "newbalanceOrig", 
          "oldbalanceDest", "newbalanceDest"]]

Y = data["isFraud"]

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=42
)

print(f'Shape of X_train:   {X_train.shape}')
print(f'Shape of X_test:    {X_test.shape}')

Y_train_fraud = Y_train.mean()
Y_test_fraud = Y_test.mean()

print(f'y_train fraud rate: {Y_train_fraud:.3%}')
print(f'y_test_fraud rate:  {Y_test_fraud:.3%}')

Shape of X_train:   (2216327, 7)
Shape of X_test:    (554082, 7)
y_train fraud rate: 0.296%
y_test_fraud rate:  0.297%


### Dummy classifier

In [ ]:
# dummy classifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, Y_train)
y_pred_dummy = dummy.predict(X_test)

print(f"Accuracy:  {accuracy_score(Y_test, y_pred_dummy):.3%}")
print(f"Recall:    {recall_score(Y_test, y_pred_dummy):.3f}")
print(f"Precision: {precision_score(Y_test, y_pred_dummy, zero_division=0):.3f}")

Accuracy:  99.703%
Recall:    0.000
Precision: 0.000


### Logistic regresssion baseline

In [39]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

numeric_cols = ["step", "amount", "oldbalanceOrg", "newbalanceOrig",
                "oldbalanceDest", "newbalanceDest"]
categorical_cols = ["type"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(), categorical_cols),
])

log_reg = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000)),
])

log_reg.fit(X_train, Y_train)
y_pred_lr = log_reg.predict(X_test)


print(f"Accuracy:  {accuracy_score(Y_test, y_pred_lr):.3%}")
print(f"Recall:    {recall_score(Y_test, y_pred_lr):.3f}")
print(f"Precision: {precision_score(Y_test, y_pred_lr, zero_division=0):.3f}")

Accuracy:  94.134%
Recall:    0.884
Precision: 0.043


- Recall 0.884: it catches 88% of fraud. The test set has about 1,640 frauds, so it catches about 1,450 and misses about 190. The dummy caught 0.

- Precision 0.043: only 4.3% of flagged transactions are really fraud. That’s about 23 genuine customers blocked for every fraud caught, roughly 34,000 flags in total. In production that would be unworkable.

- Accuracy 94.1% is lower than the dummy’s 99.7%, yet this model is far more useful. That’s the accuracy trap again, and a strong line for your README.

Why precision is so low:
class_weight="balanced" makes the model very eager to flag fraud.
The default threshold of 0.5 isn’t tuned to the bank’s costs. Choosing a better one is M2.
Logistic regression can only draw a straight-line boundary between fraud and genuine. XGBoost in M3 can learn more complex patterns.

### M1 questions

**Why stratify?**
Fraud is only 0.3% of rows. A random split could leave the test set with more or less fraud than reality, making scores unreliable. Stratifying keeps the fraud rate equal in train and test (0.296% vs 0.297%), so the test set represents real data.

**Why does a pipeline prevent leakage?**
Preprocessing learns from data. The scaler learns each column's mean and standard deviation. In a pipeline, these are learned from the training set only and then applied unchanged to the test set. Scaling the full dataset first would let test information influence training, giving over-optimistic scores.

**What does a baseline prove?**
It sets the bar a model must beat to show it has learned real signal. The dummy (always "not fraud") scores 99.7% accuracy but 0 recall, showing that accuracy is misleading. Logistic regression reaches 0.884 recall (precision 0.043), so the features carry real fraud signal. It is now the benchmark XGBoost must beat.